<a href="https://colab.research.google.com/github/Striver29/CrossEnrich/blob/main/CrossEnrich_v0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Phase 1: Hardcoded data collection**

In [1]:
!pip install gprofiler-official --quiet
print("done")

done


In [2]:
from gprofiler import GProfiler
import pandas as pd

In [3]:
gene_list = ["TP53", "MDM2", "CDKN1A", "BAX", "BBC3", "PMAIP1", "GADD45A", "GADD45B",
"SESN1", "SESN2", "RRM2B", "DRAM1", "FAS", "TNFRSF10B", "APAF1",
"CASP3", "CASP9", "CYCS", "PTEN", "ATM", "ATR", "BRCA1", "BRCA2"]

print(len(gene_list))
print(gene_list)

23
['TP53', 'MDM2', 'CDKN1A', 'BAX', 'BBC3', 'PMAIP1', 'GADD45A', 'GADD45B', 'SESN1', 'SESN2', 'RRM2B', 'DRAM1', 'FAS', 'TNFRSF10B', 'APAF1', 'CASP3', 'CASP9', 'CYCS', 'PTEN', 'ATM', 'ATR', 'BRCA1', 'BRCA2']


In [4]:
gp = GProfiler()
results = gp.profile(gene_list, organism="hsapiens", no_evidences=False) #no_evidences=False is set to show intersections
results = pd.DataFrame(results)
print(results.shape)

(608, 16)


In [5]:
results.head()

,description,effective_domain_size,intersection_size,intersections,name,native,p_value,parents,precision,query,query_size,recall,significant,source,term_size,evidences
0,p53 signaling pathway,8484,20,"[TP53, MDM2, CDKN1A, BAX, BBC3, PMAIP1, GADD45...",p53 signaling pathway,KEGG:04115,5.941134e-39,[KEGG:00000],0.909091,query_1,22,0.270270,True,KEGG,74,"[[KEGG], [KEGG], [KEGG], [KEGG], [KEGG], [KEGG..."
1,DNA damage response,8752,19,"[TP53, MDM2, CDKN1A, BAX, BBC3, PMAIP1, GADD45...",DNA damage response,WP:WP707,8.715110e-36,[WP:000000],0.826087,query_1,23,0.275362,True,WP,69,"[[WP], [WP], [WP], [WP], [WP], [WP], [WP], [WP..."
2,miRNA regulation of DNA damage response,8752,19,"[TP53, MDM2, CDKN1A, BAX, BBC3, PMAIP1, GADD45...",miRNA regulation of DNA damage response,WP:WP1530,2.994203e-35,[WP:000000],0.826087,query_1,23,0.260274,True,WP,73,"[[WP], [WP], [WP], [WP], [WP], [WP], [WP], [WP..."
3,Platinum drug resistance,8484,13,"[TP53, MDM2, CDKN1A, BAX, BBC3, PMAIP1, FAS, A...",Platinum drug resistance,KEGG:01524,1.175199e-20,[KEGG:00000],0.590909,query_1,22,0.180556,True,KEGG,72,"[[KEGG], [KEGG], [KEGG], [KEGG], [KEGG], [KEGG..."
4,miRNA regulation of p53 pathway in prostate ca...,8752,11,"[TP53, MDM2, BAX, BBC3, PMAIP1, TNFRSF10B, APA...",miRNA regulation of p53 pathway in prostate ca...,WP:WP3982,2.706305e-20,[WP:000000],0.478261,query_1,23,0.354839,True,WP,31,"[[WP], [WP], [WP], [WP], [WP], [WP], [WP], [WP..."


In [6]:
print(results["source"].unique())

['KEGG' 'WP' 'REAC' 'GO:BP' 'HP' 'GO:CC' 'MIRNA' 'CORUM' 'GO:MF' 'TF'
 'HPA']


**Note: We treat GO as three separate databases:**

GO:BP (Biological Processes)

GO:CC (Cellular Component)

GO:MF (Molecular Function)

This leaves us with 6 databases for our analysis (12 pairwise comparison): GO:BP, GO:CC, GO:MF, KEGG, WP REACTOM.

#**Phase 2: Source filteration**




In [7]:
df_gobp = results[results["source"] == "GO:BP"]
df_gocc = results[results["source"] == "GO:CC"]
df_gomf = results[results["source"] == "GO:MF"]
df_kegg = results[results["source"] == "KEGG"]
df_reac = results[results["source"] == "REAC"]
df_wp = results[results["source"] == "WP"]

The code below is a safe guard filter for allowing terms that multiple correction

In [8]:
df_gobp = df_gobp[df_gobp["significant"] == True]
df_gocc = df_gocc[df_gocc["significant"] == True]
df_gomf = df_gomf[df_gomf["significant"] == True]
df_kegg = df_kegg[df_kegg["significant"] == True]
df_reac = df_reac[df_reac["significant"] == True]
df_wp = df_wp[df_wp["significant"] == True]

In [9]:
print("GO:BP ",df_gobp.shape)
print("GO:CC ",df_gocc.shape)
print("GO:MF ", df_gomf.shape)
print("KEGG ", df_kegg.shape)
print("REAC ", df_reac.shape)
print("WP ", df_wp.shape)

GO:BP  (236, 16)
GO:CC  (12, 16)
GO:MF  (13, 16)
KEGG  (54, 16)
REAC  (76, 16)
WP  (68, 16)


#**Phase 3: Consistency and Metrics**




**Term Jaccard**



In [10]:
#mock example of two databases and direct name matching.

df_kegg_set = set(df_kegg["name"])
df_reac_set = set(df_reac["name"])

union = df_kegg_set.union(df_reac_set)
intersection = df_kegg_set.intersection(df_reac_set)

jaccard = len(intersection)/len(union)
print(jaccard)

0.007751937984496124


In [11]:
print(intersection)

#only one direct match

{'Apoptosis'}


In [12]:
#function for cross-database name direct matching.

def term_jaccard(db1, db2):
  db1_set = set(db1["name"])
  db2_set = set(db2["name"])

  union = db1_set.union(db2_set)
  intersection = db1_set.intersection(db2_set)

  if len(union) == 0:
    jaccard = 0
  else:
    jaccard = len(intersection)/len(union)

  return jaccard

In [13]:
databases = [
    ("GO:BP", df_gobp),
    ("GO:CC", df_gocc),
    ("GO:MF", df_gomf),
    ("KEGG", df_kegg),
    ("REAC", df_reac),
    ("WP", df_wp)
]

db_names = [name for name, df in databases]
matrix_df = pd.DataFrame(index=db_names, columns=db_names, dtype=float)

for i in range(len(databases)):
    name_i, df_i = databases[i]
    for j in range(i+1, len(databases)):
        name_j, df_j = databases[j]
        score = term_jaccard(df_i, df_j)
        matrix_df.at[name_i, name_j] = score
        matrix_df.at[name_j, name_i] = score

print(matrix_df)

        GO:BP  GO:CC  GO:MF      KEGG      REAC        WP
GO:BP     NaN    0.0    0.0  0.000000  0.000000  0.003300
GO:CC  0.0000    NaN    0.0  0.000000  0.000000  0.000000
GO:MF  0.0000    0.0    NaN  0.000000  0.000000  0.000000
KEGG   0.0000    0.0    0.0       NaN  0.007752  0.051724
REAC   0.0000    0.0    0.0  0.007752       NaN  0.006993
WP     0.0033    0.0    0.0  0.051724  0.006993       NaN


**Results indicate that direct name comparison yeild nearly zero consistency**

# **Gene-level Jaccard**

In [14]:
#Checks for overlap based on terms and returns the highest overlap among two terms.
#the overlaps are then averaged for a final database-database gene_level score
def gene_jaccard_term(genes1, genes2):
    set1 = set(genes1)
    set2 = set(genes2)
    union = set1.union(set2)
    if len(union) == 0:
        return 0
    return len(set1.intersection(set2)) / len(union)

def gene_jaccard(db1, db2):

  term_scores = list()
  for idx1, row1 in db1.iterrows():
    best_score = 0

    for idx2, row2 in db2.iterrows():
        score = gene_jaccard_term(row1["intersections"], row2["intersections"])

        if score >= 0.5 and score > best_score:
            best_score = score

    if best_score != 0:
      term_scores.append(best_score)


  if len(term_scores) == 0:
    return 0
  else:
    return sum(term_scores)/len(term_scores)


In [15]:
print(gene_jaccard(df_kegg, df_reac)) #About 69% agreement with gene_level agreement

0.69604630119336


In [63]:
#building the level agreement over the 5 databases

matrix_df_gene_level = pd.DataFrame(index=db_names, columns=db_names, dtype=float)

for i in range (len(databases)):
  name_i, db_i = databases[i]
  for j in range(i+1, len(databases)):
    name_j, db_j = databases[j]
    score = gene_jaccard(db_i, db_j)
    matrix_df_gene_level.at[name_i, name_j] = score
    matrix_df_gene_level.at[name_j, name_i] = score

print(matrix_df_gene_level)

          GO:BP     GO:CC     GO:MF      KEGG      REAC        WP
GO:BP       NaN  0.638011  0.594935  0.634765  0.646506  0.657019
GO:CC  0.638011       NaN  0.560965  0.697826  0.683779  0.676864
GO:MF  0.594935  0.560965       NaN  0.574838  0.617851  0.671672
KEGG   0.634765  0.697826  0.574838       NaN  0.696046  0.815816
REAC   0.646506  0.683779  0.617851  0.696046       NaN  0.781863
WP     0.657019  0.676864  0.671672  0.815816  0.781863       NaN


# **Spearman Correlation**

In [83]:
from scipy.stats import spearmanr
def gene_jaccard_term(genes1, genes2):
    set1 = set(genes1)
    set2 = set(genes2)
    union = set1.union(set2)
    if len(union) == 0:
        return 0
    return len(set1.intersection(set2)) / len(union)

def spearman_correlation(db1, db2):

    db1_pvals = []
    db2_pvals = []

    for idx1, row1 in db1.iterrows():
        best_score = 0
        best_pval = None

        for idx2, row2 in db2.iterrows():
            score = gene_jaccard_term(row1["intersections"], row2["intersections"])
            if score >= 0.5 and score > best_score:
                best_score = score
                best_pval = row2["p_value"]

        if best_pval is not None:
            db1_pvals.append(row1["p_value"])
            db2_pvals.append(best_pval)

    if len(db1_pvals) < 5: #Spearman breaks with small agreements, hence, not worth noting if we have less than 5 matches
        return None
    try:
      spearman = spearmanr(db1_pvals, db2_pvals)
      return spearman.correlation
    except Exception:
      return None


In [28]:
print(spearman_correlation(df_kegg, df_reac))

0.37927918929484117


In [29]:
matrix_df_spearman = pd.DataFrame(index=db_names, columns=db_names, dtype=float)

for i in range (len(databases)):
  name_i, db_i = databases[i]
  for j in range (i+1, len(databases)):
    name_j, db_j = databases[j]

    score = spearman_correlation(db_i, db_j)
    matrix_df_spearman.at[name_i, name_j] = score
    matrix_df_spearman.at[name_j, name_i] = score

print(matrix_df_spearman)

          GO:BP     GO:CC     GO:MF      KEGG      REAC        WP
GO:BP       NaN  0.002083  0.058833  0.435762  0.330438  0.457200
GO:CC  0.002083       NaN  0.925820 -0.065290 -0.317887 -0.202784
GO:MF  0.058833  0.925820       NaN  0.738769  0.201688 -0.139818
KEGG   0.435762 -0.065290  0.738769       NaN  0.379279  0.602388
REAC   0.330438 -0.317887  0.201688  0.379279       NaN  0.416784
WP     0.457200 -0.202784 -0.139818  0.602388  0.416784       NaN


In [57]:
def parse_gmt(filename):
  with open(filename, "r") as f:
    lines = f.readlines()
    lines = lines[0]
    lines = lines.split("\t")
    res = list()
    for i in range (2, len(lines)):
      res.append(lines[i].strip())

    return (lines[0], res)

In [61]:
gmt_list = ["HALLMARK_ANGIOGENESIS.v2026.1.Hs.gmt",
            "HALLMARK_DNA_REPAIR.v2026.1.Hs.gmt",
            "HALLMARK_OXIDATIVE_PHOSPHORYLATION.v2026.1.Hs.gmt",
            "HALLMARK_P53_PATHWAY.v2026.1.Hs.gmt",
            "HALLMARK_TNFA_SIGNALING_VIA_NFKB.v2026.1.Hs.gmt"]

gene_dict = {}

for gmt in gmt_list:
  name, genes = parse_gmt(gmt)
  gene_dict[name] = genes
  print("Name", name)
  print("Gene Count", len(genes))
  print("\n")


Name HALLMARK_ANGIOGENESIS
Gene Count 36


Name HALLMARK_DNA_REPAIR
Gene Count 150


Name HALLMARK_OXIDATIVE_PHOSPHORYLATION
Gene Count 200


Name HALLMARK_P53_PATHWAY
Gene Count 200


Name HALLMARK_TNFA_SIGNALING_VIA_NFKB
Gene Count 200




In [76]:
def run_pipeline(gene_list):
  #instantiate and run g:profiler
  gp = GProfiler()
  results = gp.profile(gene_list, organism="hsapiens", no_evidences=False) #no_evidences=False is set to show intersections
  results = pd.DataFrame(results)


  #Source Filtration
  df_gobp = results[results["source"] == "GO:BP"]
  df_gocc = results[results["source"] == "GO:CC"]
  df_gomf = results[results["source"] == "GO:MF"]
  df_kegg = results[results["source"] == "KEGG"]
  df_reac = results[results["source"] == "REAC"]
  df_wp = results[results["source"] == "WP"]


  #Multiple Correction Safeguard
  df_gobp = df_gobp[df_gobp["significant"] == True]
  df_gocc = df_gocc[df_gocc["significant"] == True]
  df_gomf = df_gomf[df_gomf["significant"] == True]
  df_kegg = df_kegg[df_kegg["significant"] == True]
  df_reac = df_reac[df_reac["significant"] == True]
  df_wp = df_wp[df_wp["significant"] == True]

  #Set up environment for metric evaluation
  databases = [
    ("GO:BP", df_gobp),
    ("GO:CC", df_gocc),
    ("GO:MF", df_gomf),
    ("KEGG", df_kegg),
    ("REAC", df_reac),
    ("WP", df_wp)
  ]

  db_names = [name for name, df in databases]

  #Run Term_level Jaccard first
  matrix_term_jaccard = pd.DataFrame(index=db_names, columns=db_names, dtype=float)

  for i in range(len(databases)):
    name_i, df_i = databases[i]
    for j in range(i+1, len(databases)):
        name_j, df_j = databases[j]
        score = term_jaccard(df_i, df_j)
        matrix_term_jaccard.at[name_i, name_j] = score
        #matrix_term_jaccard.at[name_j, name_i] = score


  #Run Gene_level Jaccard
  matrix_gene_level_jaccard = pd.DataFrame(index=db_names, columns=db_names, dtype=float)

  for i in range (len(databases)):
    name_i, db_i = databases[i]
    for j in range(i+1, len(databases)):
      name_j, db_j = databases[j]
      score = gene_jaccard(db_i, db_j)
      matrix_gene_level_jaccard.at[name_i, name_j] = score
      #matrix_gene_level_jaccard.at[name_j, name_i] = score

  #Run Spearman_Correlation
  matrix_spearman = pd.DataFrame(index=db_names, columns=db_names, dtype=float)

  for i in range (len(databases)):
    name_i, db_i = databases[i]
    for j in range (i+1, len(databases)):
      name_j, db_j = databases[j]

      score = spearman_correlation(db_i, db_j)
      matrix_spearman.at[name_i, name_j] = score
      #matrix_spearman.at[name_j, name_i] = score

  #return three matrices
  return matrix_term_jaccard, matrix_gene_level_jaccard, matrix_spearman


In [82]:
for idx in gene_dict:
  matrices = run_pipeline(gene_dict[idx])
  print("Gene List:", idx)
  print("\n")
  print("Term Jaccard:\n", matrices[0])
  print("\n")
  print("Gene_level Jaccard:\n", matrices[1])
  print("\n")
  print("Spearman Correlation:\n",matrices[2])
  print("\n\n")


Gene List: HALLMARK_ANGIOGENESIS


Term Jaccard:
        GO:BP  GO:CC  GO:MF  KEGG  REAC        WP
GO:BP    NaN    0.0    0.0   0.0   0.0  0.000000
GO:CC    NaN    NaN    0.0   0.0   0.0  0.000000
GO:MF    NaN    NaN    NaN   0.0   0.0  0.000000
KEGG     NaN    NaN    NaN   NaN   0.0  0.076923
REAC     NaN    NaN    NaN   NaN   NaN  0.000000
WP       NaN    NaN    NaN   NaN   NaN       NaN


Gene_level Jaccard:
        GO:BP     GO:CC     GO:MF      KEGG      REAC        WP
GO:BP    NaN  0.594033  0.648570  0.542424  0.568122  0.587838
GO:CC    NaN       NaN  0.610218  0.000000  0.608225  0.000000
GO:MF    NaN       NaN       NaN  0.535714  0.563492  0.551852
KEGG     NaN       NaN       NaN       NaN  0.500000  0.731111
REAC     NaN       NaN       NaN       NaN       NaN  0.533670
WP       NaN       NaN       NaN       NaN       NaN       NaN


Spearman Correlation:
        GO:BP    GO:CC     GO:MF      KEGG      REAC        WP
GO:BP    NaN  0.27055  0.162307 -0.187088 -0.216615  0.3

# **Important Observations:**

**1. Gene_level Jaccard gives high average agreement across Databases (ranges between 50%-70% on every pair)**

**2. While Gene_level results were somehow consistent, spearman results were completely unstable. This shows that databases might agree on the genes involved, yet, not their significance**

**3.Oxidative Phosphorylation is the most consistent gene set in both gene_level and spearman correlation. This is because Oxidative Phosphorylation is one of the most ancient and well-established biological processes of all time, hence, most databases are consistent with respect to this specific set. This generates an interesing observation: consistency correlates with biological certainty.**


**4. Smaller gene sets like Angiogenesis generates fewer matches across databases. Small gene sets yeild few enriched terms which generates fewer matches which yeilds to insignificant spearman results, hence, our spearman function returns None if less than 5 matches are found.**


